# Comparative Evaluation of Total RNA-Seq and mRNA-Seq Pipelines in Bulk Transcriptomics

> - Author: Cao Qiang
> - Email: qiangcao2014@gmail.com
> - Created: 2022-03-28
> - Modified: 2022-03-28

In [ ]:
data_dir <- "~/reference/Mus_musculus/mm10"
# load reference gene anntotion
gtfDir <- file.path(data_dir, "gencode.vM23.annotation.gtf")
gtf <- rtracklayer::import(gtfDir)
names(gtf) <- gtf$gene_id

gtf |> 
    subset(type == "gene") |>
    subset(gene_type == "protein_coding") |>
    granges() |>
    rtracklayer::export.bed(
        file.path(data_dir, "mm10_vM23_proteinCoding.bed"))

gtf |> 
    subset(type == "gene") |>
    subset(gene_type == "lncRNA") |>
    granges() |>
    rtracklayer::export.bed(
        file.path(data_dir, "mm10_vM23_lncRNA.bed"))

In [ ]:
# suppressPackageStartupMessages(library(GenomicFeatures))

txdb <- txdbmaker::makeTxDbFromGFF(
    gtfDir, dataSource = "Gencode", 
    organism = "Mus musculus", taxonomyId = 10090, 
    metadata = data.frame(
        name=c("Genome", "Gencode Release"), 
        value=c("mm10", "vM23")))

# txdb <- txdbmaker::makeTxDbFromGRanges(
#     gtf, drop.stop.codons = TRUE, taxonomyId = 10090, 
#     metadata = data.frame(
#         name=c("Genome", "Gencode Release"),
#         value=c("mm10", "vM23")))

In [ ]:
Sys.time()
exons_length <- gtf |> 
    subset(type == "exon") |>
    split(~ gene_id) |>
    map(reduce) |>
    map_int(function(gr) sum(width(gr))) |>
    enframe(name = "gene_id", value = "basepairs")
Sys.time()

In [ ]:
Sys.time()
exons_length <- gtf |> 
    subset(type == "exon") |>
    split(~ gene_id) |>
    lapply(reduce) |>
    sapply(FUN = function(gr) sum(width(gr))) |>
    enframe(name = "gene_id", value = "basepairs")
Sys.time()

In [ ]:
geneAnno <- gtf |>
    subset(type == "gene") |>
    subset(gene_type %in% c("protein_coding", "lncRNA")) |>
    as.data.frame() |>
    distinct(chr = seqnames, start, end, strand, 
             gene_id, symbol = gene_name, type = gene_type) |>
    left_join(exons_length, by = "gene_id") |>
    mutate(gene_id = sub("\\..*$", "", gene_id)) |>
    column_to_rownames("gene_id") |> 
    DataFrame()

In [ ]:
suppressPackageStartupMessages(library(org.Hs.eg.db))

In [ ]:
gene_map <- mapIds(org.Hs.eg.db, keys = rownames(countMat), 
                   column = "SYMBOL", keytype = "ENSEMBL", 
                   multiVals = "first") |> 
    data.frame(mgi_symbol = _) |> 
    rownames_to_column("ensembl_gene_id")
head(gene_map)

In [ ]:
suppressPackageStartupMessages(library(biomaRt))

In [ ]:
# ensembl <- useEnsembl(biomart = "genes", 
#                       dataset = "hsapiens_gene_ensembl",
#                       mirror = "useast")
ensembl <- useEnsembl(biomart = "genes", 
                      dataset = "mmusculus_gene_ensembl",
                      verbose = TRUE)

gene_map <- getBM(attributes = c("ensembl_gene_id", "mgi_symbol"),
                  mart = ensembl, filters = "ensembl_gene_id", 
                  values = rownames(countMat))
head(gene_map)